In [20]:
from typing import List
import numpy as np
import stim
import networkx as nx
from stimcirq import stim_circuit_to_cirq_circuit, cirq_circuit_to_stim_circuit
import cirq
import openfermion as of
from encoded.code_extension import encoding_unitary_for_new_stabilizer
from encoded.utils import cirq_pauli_string_to_stim, stim_pauli_string_to_cirq
from encoded.diagonalizing_circuit import get_measurement_circuit

Start with the group $\langle ZIZIZIZI, IZIZIZIZ \rangle$ and add stabilizers to correct all bit-flip errors.

In [21]:
generators = [
    stim.PauliString("ZIZIZIZIII"),
    stim.PauliString("IZIZIZIZII"),
    stim.PauliString("ZZIIZZIIZI"),
    stim.PauliString("ZIZIIZIZIZ")
]

In [22]:
def all_bit_flip_errors(n: int) -> List[stim.PauliString]:
    errs = []
    for i in range(n):
        pauli_mask = [0] * i + [1] + [0] * (n - i - 1)
        errs.append(stim.PauliString(pauli_mask))
    return errs

In [23]:
errors = all_bit_flip_errors(8)
for err in errors:
    print(err)

+X_______
+_X______
+__X_____
+___X____
+____X___
+_____X__
+______X_
+_______X


In [24]:
def get_uncorrectable_errors(generators):
    number_true = 0
    number_checked = 0
    uncorrectable_errors = []
    for i, ei in enumerate(errors):
        for j in range(i):
            number_checked += 1
            ej = errors[j]
            e = ei * ej
            commutators = []
            for generator in generators:
                comm = e.commutes(generator)
                commutators.append(comm)
            has_anticommuting_operator = any([not b for b in commutators])
            if has_anticommuting_operator:
                number_true += 1
            else:
                print(f"{ei} * {ej} = {e}, {commutators} {has_anticommuting_operator} ")
                uncorrectable_errors.append(e)
    print(f"{number_true}/{number_checked} operators anticommute.")
    return uncorrectable_errors

In [25]:
bad_errors = get_uncorrectable_errors(generators)

28/28 operators anticommute.


So two extra, noiseless qubits are needed.